In [2]:
%pip install PyMuPDF
%pip install langchain_text_splitters


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
from langchain_core.documents import Document
import os
from rag.embedding_model import _bedrock_embeddings
import fitz
from langchain_text_splitters import RecursiveCharacterTextSplitter
from IPython.display import clear_output
import dotenv
import uuid


In [25]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

In [4]:
dotenv.load_dotenv()

True

In [5]:
from langchain_postgres import PGEngine

pg_engine = PGEngine.from_connection_string(url=os.environ['ASYNC_DATABASE_URL'])


In [6]:
await pg_engine.ainit_vectorstore_table(
    table_name='DOCUMENT_VECTOR_DB',
    vector_size=1024,
)

In [ ]:
from langchain_postgres import PGVectorStore

store = await PGVectorStore.create(
    engine=pg_engine,
    table_name='DOCUMENT_VECTOR_DB',
    # schema_name=SCHEMA_NAME,
    embedding_service=_bedrock_embeddings,
)


In [62]:
from warnings import filterwarnings
files = os.listdir('Data/Acts')
base = 'Data/Acts'
BATCH_SIZE = 20

for file in files:
    print(f'Processing {file} ....')
    doc = fitz.open(os.path.join(base, file))
    name = file.split('.')[0]
    txt = ''

    for i in range(len(doc)):
        txt += doc.get_page_text(i)
    
    docs = splitter.split_documents([
        Document(
            metadata = {
                'source': name,
            },
            page_content = txt
        )
    ])

    for i, doc in enumerate(docs):
        doc.metadata['chunk_id']=str(uuid.uuid4())
        doc.metadata['chunk_index']=i+1
    
    for i in range(0, len(docs), BATCH_SIZE):
        clear_output(wait=True)
        print(f'Processing {file} ....')
        print(f'Processing {i//BATCH_SIZE + 1}/{len(docs)//BATCH_SIZE + 1} Batch!')
        batch = docs[i:i+BATCH_SIZE]
        await store.aadd_documents(batch)
        
    

Processing IT.pdf ....
Processing 13/13 Batch!


In [51]:
docs[0].metadata

{'source': 'BNSS'}